# Options Explorer

Fetch option chains, Greeks, and implied volatility surfaces for SPY, VIX, and commodity ETFs.

This notebook works in two modes:
- **Live** — connected to IB Gateway / TWS (real-time data)
- **Fallback** — no TWS needed (CBOE delayed quotes / yfinance)

In [ ]:
from ibkr_eda.options.chain import OptionChains
from ibkr_eda.options.greeks import Greeks
from ibkr_eda.options.surface import VolSurface
from ibkr_eda.options.fallback_provider import FallbackOptionsProvider

# Use fallback provider (no TWS connection required)
provider = FallbackOptionsProvider()

options = OptionChains(provider=provider)
greeks = Greeks(provider=provider)
vol_surface = VolSurface(provider=provider)

# Alternatively, with a live TWS connection:
# from ibkr_eda import IBKR
# ib = IBKR()
# options = ib.options
# greeks = ib.greeks
# vol_surface = ib.vol_surface

## 1. Available Expirations

In [ ]:
spy_expiries = options.get_expirations("SPY")
print(f"SPY has {len(spy_expiries)} expiry dates")
print("Nearest 10:", spy_expiries[:10])

## 2. Fetch SPY Option Chain

In [ ]:
# Pick the nearest monthly expiry
expiry = spy_expiries[0]
print(f"Fetching chain for SPY {expiry}...")

chain = options.get("SPY", expiry)
print(f"Underlying price: {chain.underlying_price}")
print(f"Calls: {len(chain.calls)} rows")
print(f"Puts:  {len(chain.puts)} rows")

In [ ]:
chain.calls.head(10)

In [ ]:
chain.puts.head(10)

## 3. Greeks for a Single Contract

In [ ]:
# Pick an ATM-ish strike
atm_strike = round(chain.underlying_price / 5) * 5  # round to nearest 5
print(f"ATM strike: {atm_strike}")

greek_df = greeks.get("SPY", expiry, atm_strike, "C")
greek_df.T

## 4. Implied Volatility Smile / Skew

In [ ]:
import plotly.express as px

skew = vol_surface.get_skew("SPY", expiry)
skew = skew.dropna(subset=["call_iv", "put_iv"], how="all")

fig = px.line(
    skew, x="strike", y=["call_iv", "put_iv"],
    title=f"SPY IV Skew — {expiry}",
    labels={"value": "Implied Volatility", "strike": "Strike"},
)
fig.show()

## 5. IV Surface (3D)

In [ ]:
import plotly.graph_objects as go

surface_data = vol_surface.get_raw("SPY")

fig = go.Figure(data=[go.Surface(
    z=surface_data.call_iv,
    x=surface_data.strikes,
    y=list(range(len(surface_data.expiries))),
    colorscale="Viridis",
)])
fig.update_layout(
    title="SPY Call IV Surface",
    scene=dict(
        xaxis_title="Strike",
        yaxis_title="Expiry Index",
        zaxis_title="Implied Vol",
    ),
)
fig.show()

## 6. VIX Options

In [ ]:
vix_expiries = options.get_expirations("VIX")
print(f"VIX expiries: {vix_expiries[:6]}")

vix_chain = options.get("VIX", vix_expiries[0])
print(f"VIX underlying: {vix_chain.underlying_price}")
vix_chain.calls.head()

## 7. ATM IV Term Structure

In [ ]:
term = vol_surface.get_term_structure("SPY")

fig = px.line(
    term, x="expiry", y=["call_iv", "put_iv"],
    title="SPY ATM IV Term Structure",
    labels={"value": "Implied Volatility", "expiry": "Expiration"},
)
fig.show()

## 8. Commodity ETF Options (GLD, USO)

In [ ]:
for sym in ["GLD", "USO"]:
    exp = options.get_expirations(sym)
    if exp:
        chain = options.get(sym, exp[0])
        print(f"\n{sym} — expiry {exp[0]}, underlying {chain.underlying_price}")
        print(f"  Calls: {len(chain.calls)}, Puts: {len(chain.puts)}")
        display(chain.calls[["strike", "last", "bid", "ask", "implied_vol", "delta"]].head(5))

## 9. Flat DataFrame (all calls + puts)

In [ ]:
flat_df = options.get_df("SPY", spy_expiries[0])
print(f"Flat chain shape: {flat_df.shape}")
flat_df.head()